# 14 · Dense FIMI datasets: out-of-sample test of the cost model  (VM; lite ≈ 30–60 min)

**Why.** All 18 paper datasets are sparse (density d_avg/N ≤ 2.6 %) and have P1 ≥ 23 at s = 1e-4;
scatter beat classic on every one of the 17 correlation points (1.6–413×). The 0.95 correlation
therefore shows that the *size* of the speedup follows P1, but never tests the *crossover*. On the
dense half of the FIMI repository (chess 49 %, connect 33 %, mushroom 19 %, accidents 7 %, pumsb
3.5 % density) P1 is about 0.5–8, i.e. d_avg(1+d_CV²) approaches |F1|, where the model predicts that
candidate intersection (`classic`, Eclat's level 2) matches or beats scatter. This notebook tests that
prediction on data the model was not developed on. Either outcome is reportable.

**Lite design (default).**

| part | datasets | supports | k | methods |
|---|---|---|---|---|
| A · cost model only | all five | 0.5 … 1e-3 | – | P1, F1, density (no mining) |
| B · pairwise crossover | all five | 0.5, 0.1, 0.05, 0.01, 0.005, 1e-3 | 2 | `hyb_1t`, `classic_1t`, `hyb`, `classic`, `borgelt_apriori`, `borgelt_eclat` |
| C · higher order (optional) | chess, mushroom, connect | same | 3 | `hyb`, `hyb_1t`, `borgelt_eclat`, `borgelt_fpgrowth`, `lcm` |

Supports stop at **1e-3** (author decision, 29 Oct: 1e-4 is not feasible on dense data). The grid
runs from high to low support and, per method, stops descending after its first timeout/OOM
(recorded, not hidden). Classic and scatter are compared at **matched threads** (1/1 and 8/8), so the
thread question of the 0.95 does not arise here.

`SMOKE = True` runs chess at two supports, one measured run, into a separate experiment
(`fimi_lite_smoke`) to check the setup in a minute. `RUN_FULL = True` runs the full grid
(nine k = 2 methods, k = 3–4, supports to 1e-4); it is open-ended and OOM-prone.

Output: `results/runs/<campaign>/fimi_lite.csv` (smoke: `fimi_lite_smoke.csv`; full: `fimi_dense.csv`),
plus `fimi_lite_crossover.csv` / `.png`.

`FA_MODE=replay` analyses the saved campaign (`vm_20261029` unless `FA_CAMPAIGN` says otherwise): no mining and no data needed; tables and figures go to `results/derived/<campaign>/`.


In [ ]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
REPLAY = os.environ.get("FA_MODE") == "replay"    # analyse the saved campaign: no mining, nothing overwritten
os.environ.setdefault("FA_CAMPAIGN", "vm_20261029" if REPLAY else "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
OUT = BUNDLE / "results" / "derived" / pr.CAMPAIGN if REPLAY else pr.RESULTS
OUT.mkdir(parents=True, exist_ok=True)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

In [ ]:
SMOKE = False          # True: chess, two supports, one measured run, experiment "fimi_lite_smoke"
RUN_K3 = True          # part C
RUN_FULL = False       # the full open-ended grid (supports to 1e-4, k = 3-4, nine methods)

FIMI = ["chess", "mushroom", "connect", "accidents", "pumsb"]          # small -> large
SUPPORTS = [0.5, 0.1, 0.05, 0.01, 0.005, 1e-3]                          # high -> low
K2_METHODS = ["hyb_1t", "classic_1t", "hyb", "classic", "borgelt_apriori", "borgelt_eclat"]
K3_DATASETS = ["chess", "mushroom", "connect"]
K3_METHODS = ["hyb", "hyb_1t", "borgelt_eclat", "borgelt_fpgrowth", "lcm"]
REPEATS, WARMUP, TIMEOUT_S = 3, 1, 600
EXPERIMENT = "fimi_lite"
if SMOKE:
    FIMI, K3_DATASETS, SUPPORTS = ["chess"], ["chess"], [0.5, 0.1]
    REPEATS, WARMUP, TIMEOUT_S, EXPERIMENT = 1, 0, 120, "fimi_lite_smoke"
pr.RESULTS.mkdir(parents=True, exist_ok=True)
print(f"{EXPERIMENT}: {FIMI} x {SUPPORTS}, repeats={REPEATS}, warm-up={WARMUP}, timeout={TIMEOUT_S}s")

def descend(dataset, methods, ks, experiment=EXPERIMENT, supports=SUPPORTS, timeout_s=TIMEOUT_S,
            repeats=REPEATS, warmup=WARMUP):
    """Run supports high -> low; a method that fails (timeout/oom) at support s for the smallest k
    is not run at lower supports (more work), and the failure stays in the CSV."""
    alive = list(methods)
    for s in supports:
        if not alive:
            break
        res = pr.run_grid(experiment, [dataset], alive, ks=ks, supports=[s], timeout_s=timeout_s,
                          repeats=repeats, warmup=warmup)
        cell = res[(res.dataset == dataset) & np.isclose(res.min_support, s) & (res.k == min(ks))]
        dead = set(cell[cell.status.isin(["timeout", "oom"])].method)
        if dead:
            print(f"{dataset} s={s:g}: stop descending for {sorted(dead)}")
        alive = [m for m in alive if m not in dead]

## A · Cost-model statistics (no mining)

P1 = (|F1| − 1) / (d_avg (1 + d_CV²)) per dataset and support, as in the paper (raw widths). The
17 correlation datasets have P1 between 23 and 2,811 at s = 1e-4.

In [ ]:
if REPLAY:
    cost = pd.read_csv(pr.RESULTS / f"{EXPERIMENT}_cost_stats.csv")
else:
    rows = []
    for n in FIMI:
        df = pdd.fetch_fimi(n)
        for st in pdd.cost_stats(df, SUPPORTS):
            rows.append(dict(dataset=n, min_support=st["min_support"], D=st["D"], N=st["N"], d_avg=st["d_avg"],
                             density=st["d_avg"] / st["N"], d_cv=st["d_cv"], F1=st["F1"], P1=st["P1_raw"]))
    cost = pd.DataFrame(rows)
    cost.to_csv(pr.RESULTS / f"{EXPERIMENT}_cost_stats.csv", index=False)
display(cost.round(3))


## B · Pairwise crossover (k = 2)

In [ ]:
if not REPLAY:
    pr.preflight(K2_METHODS)
    for n in FIMI:
        descend(n, K2_METHODS, ks=[2])

## C · Higher order (k = 3, optional)

HybridFIM is expected to be weakest here (dense lattices); the small datasets only.

In [ ]:
if RUN_K3 and not REPLAY:
    pr.preflight(K3_METHODS)
    for n in K3_DATASETS:
        descend(n, K3_METHODS, ks=[3])

## Results

Median wall time per cell, failures, count agreement, and the crossover test: classic ÷ scatter at
matched threads against P1. A ratio below 1 means candidate intersection won. Chess and mushroom are
small (3k and 8k transactions): many of their runs take milliseconds, where fixed per-call costs
dominate, so a ratio counts toward the test only if both runs take at least `MIN_RESOLVED_S`
(unresolved cells are shown hollow and listed, not dropped silently).

In [ ]:
raw = pr.load_results(EXPERIMENT)
summ = pr.summarize(raw)
fails = raw[raw.status != "ok"].groupby(["dataset", "k", "min_support", "method", "status"]).size()
print("failed / skipped cells:", "none" if fails.empty else ""); display(fails.to_frame("runs")) if not fails.empty else None
bad = pr.check_agreement(summ)
print("count disagreements:", len(bad)); display(bad) if len(bad) else None
for k in sorted(summ.k.unique()):
    w = summ[summ.k == k].pivot_table(index=["dataset", "min_support"], columns="method", values="wall_s")
    print(f"k = {k}: median wall s"); display(w.round(3))

In [ ]:
MIN_RESOLVED_S = 0.1   # a ratio counts only if both runs take at least this long (else fixed costs dominate)
w2 = summ[summ.k == 2].pivot_table(index=["dataset", "min_support"], columns="method", values="wall_s")
x = cost.set_index(["dataset", "min_support"])[["density", "F1", "P1"]].join(w2, how="inner")
PAIRS = {"classic/scatter 1T": ("classic_1t", "hyb_1t"), "classic/scatter 8T": ("classic", "hyb")}
for col, (num, den) in PAIRS.items():
    if {num, den} <= set(x.columns):
        x[col] = x[num] / x[den]
        x[col + " resolved"] = np.minimum(x[num], x[den]) >= MIN_RESOLVED_S
ext = [c for c in ("borgelt_apriori", "borgelt_eclat") if c in x.columns]
if ext and "hyb_1t" in x.columns:
    x["best Borgelt / scatter 1T"] = x[ext].min(axis=1) / x["hyb_1t"]
ratio_cols = [c for c in x.columns if "/" in c and not c.endswith("resolved")]
x.to_csv(OUT / f"{EXPERIMENT}_crossover.csv")
display(x[["density", "F1", "P1"] + ratio_cols + [c for c in x.columns if c.endswith("resolved")]].round(3))

from scipy import stats
for c in PAIRS:
    if c not in x.columns:
        continue
    v = x.loc[x[c + " resolved"], ["P1", c]].dropna()
    print(f"{c}: {len(v)} of {x[c].notna().sum()} cells resolved (both >= {MIN_RESOLVED_S}s)")
    if len(v) >= 3:
        rho = stats.spearmanr(v.P1, v[c])[0]
        print(f"   Spearman(P1, ratio) = {rho:.3f}; cells where scatter loses (ratio < 1): {int((v[c] < 1).sum())}; "
              f"lowest-P1 resolved cell: P1 = {v.sort_values('P1').P1.iloc[0]:.2f}, ratio = {v.sort_values('P1')[c].iloc[0]:.3f}")

The figure puts the FIMI cells next to the 17 May correlation datasets (classic-8T ÷ scatter-8T,
k = 2, s = 1e-4, the matched-thread version of the published 0.95) on the same axes.

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID, SURFACE = "#1f1f1d", "#6b6a64", "#e4e3dd", "#fcfcfb"
C_MAY, C_8T, C_1T = "#2a78d6", "#eb6834", "#1baf7a"     # categorical slots 1-3 (validated)

may = pd.read_csv(BUNDLE / "results" / "campaign_2026-05" / "results_rigorous_20260523_115442.csv")
may = may[(may.k == 2) & np.isclose(may.min_support, 1e-4) & (may.wall_s_n_ok == may.wall_s_n_runs)]
mw = may.pivot_table(index="dataset", columns="method", values="wall_s_median")[["fastapriori_classic", "fastapriori_fast"]].dropna()
cpath = BUNDLE / "results" / "derived" / "cost_stats.csv"
mc = pd.read_csv(cpath if cpath.exists() else BUNDLE / "results" / "campaign_2026-05" / "cost_stats_20261022.csv")
mc = mc[np.isclose(mc.min_support, 1e-4)].set_index("dataset")
mj = mc[["P1_raw"]].join((mw.fastapriori_classic / mw.fastapriori_fast).rename("ratio"), how="inner")

fig, ax = plt.subplots(figsize=(7.2, 4.6), facecolor=SURFACE); ax.set_facecolor(SURFACE)
ax.scatter(mj.P1_raw, mj.ratio, s=40, c=C_MAY, edgecolors=SURFACE, linewidths=1.5, label=f"May 2026, 17 datasets, 8T/8T", zorder=3)
for col, color, marker, lab in (("classic/scatter 8T", C_8T, "s", "FIMI dense, 8T/8T"),
                                ("classic/scatter 1T", C_1T, "D", "FIMI dense, 1T/1T")):
    if col in x.columns:
        v = x[["P1", col, col + " resolved"]].dropna()
        r = v[v[col + " resolved"].astype(bool)]
        u = v[~v[col + " resolved"].astype(bool)]
        ax.scatter(r.P1, r[col], s=40, c=color, marker=marker, edgecolors=SURFACE, linewidths=1.5, label=lab, zorder=3)
        ax.scatter(u.P1, u[col], s=40, marker=marker, facecolors="none", edgecolors=color, linewidths=1.2,
                   label=f"{lab}, < {MIN_RESOLVED_S}s (not resolved)" if len(u) else None, zorder=3)
ax.axhline(1.0, color=INK, linestyle=(0, (4, 3)), linewidth=1.0)
ax.set_xscale("log"); ax.set_yscale("log")
ax.text(0.02, 0.97, "classic faster below the dashed line (ratio = 1)", transform=ax.get_yaxis_transform(),
        va="top", color=MUTED, fontsize=8)
ax.set_xlabel("P1 = (|F1| − 1) / (d_avg (1 + d_CV²))", color=INK)
ax.set_ylabel("classic ÷ scatter runtime (k = 2)", color=INK)
ax.grid(color=GRID, linewidth=0.6); ax.set_axisbelow(True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.tick_params(colors=MUTED); ax.legend(frameon=False, fontsize=8, labelcolor=INK)
fig.tight_layout()
fig.savefig(OUT / f"{EXPERIMENT}_crossover.png", dpi=200, facecolor=SURFACE)
plt.show()

## Full grid (optional, open-ended)

Nine k = 2 methods and eight k = 3–4 methods, supports 0.1 → 1e-4, experiment
`fimi_dense`. Low supports explode (pumsb: C(|F1|, 3) ≈ 8.7e8 candidates at 1e-4); keep
`FA_MEMOUT_GB` below the VM's RAM.

In [ ]:
if RUN_FULL:
    FULL_SUPPORTS = [0.1, 0.05, 0.01, 0.005, 1e-3, 5e-4, 1e-4]
    FULL_K2 = ["hyb_1t", "hyb_sfx_1t", "hyb", "classic", "borgelt_apriori", "borgelt_eclat",
               "pyfim_apriori", "pyfim_eclat", "hyb_seeded_1t"]
    FULL_K34 = ["hyb", "hyb_seeded", "hyb_seeded_1t", "borgelt_eclat", "borgelt_fpgrowth",
                "pyfim_eclat", "pyfim_fpgrowth", "lcm"]
    for n in ["chess", "mushroom", "connect", "pumsb", "accidents"]:
        descend(n, FULL_K2, ks=[2], experiment="fimi_dense", supports=FULL_SUPPORTS, timeout_s=900, repeats=3, warmup=1)
    for n in ["chess", "mushroom", "connect", "pumsb", "accidents"]:
        descend(n, FULL_K34, ks=[3, 4], experiment="fimi_dense", supports=FULL_SUPPORTS, timeout_s=900, repeats=3, warmup=1)
    s = pr.summarize(pr.load_results("fimi_dense"))
    display(s[s.k == 2].pivot_table(index=["dataset", "min_support"], columns="method", values="wall_s").round(3))
    display(pr.check_agreement(s))